# Supervised Fine-Tuning (SFT) with NVIDIA NeMo Automodel on Amazon SageMaker Training Jobs

This notebook demonstrates how to run SFT using [NVIDIA NeMo Automodel](https://github.com/NVIDIA-NeMo/Automodel) on SageMaker.

NeMo Automodel is a PyTorch DTensor-native SPMD training library that provides:
- **FSDP2** with DTensor (more efficient than FSDP1)
- **Tensor, Pipeline, Context, and Sequence Parallelism** via config
- **FP8 mixed precision** support
- **Sequence packing** for better GPU utilization
- **LoRA/PEFT** out of the box
- YAML-driven configuration with CLI overrides

## Prerequisites
### Setup and dependencies

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

In [ ]:
import os

model_id = "mistralai/Mistral-7B-v0.1"

os.environ["model_id"] = model_id

---
## NeMo Automodel Configuration

This test uses the [Mistral 7B PEFT config](https://github.com/NVIDIA-NeMo/Automodel/blob/main/examples/llm_finetune/mistral/mistral_7b_squad_peft.yaml) from the NeMo Automodel repo.

It uses the `rajpurkar/squad` dataset directly from HuggingFace Hub (no S3 data channels needed), with LoRA PEFT on FSDP2.

In [ ]:
%%bash

cat > ./args.yaml <<'EOF'
# NeMo Automodel config — Mistral 7B PEFT (LoRA) with Squad dataset
# Based on: examples/llm_finetune/mistral/mistral_7b_squad_peft.yaml

step_scheduler:
  global_batch_size: 16
  local_batch_size: 2
  ckpt_every_steps: 50
  val_every_steps: 50
  max_steps: 100

dist_env:
  backend: nccl
  timeout_minutes: 10

rng:
  _target_: nemo_automodel.components.training.rng.StatefulRNG
  seed: 1111
  ranked: true

model:
  _target_: nemo_automodel.NeMoAutoModelForCausalLM.from_pretrained
  pretrained_model_name_or_path: mistralai/Mistral-7B-v0.1
  torch_dtype: bf16

checkpoint:
  enabled: true
  checkpoint_dir: /opt/ml/checkpoints/
  model_save_format: safetensors
  save_consolidated: true

distributed:
  strategy: fsdp2
  dp_size: none
  tp_size: 1
  cp_size: 1
  sequence_parallel: false

loss_fn:
  _target_: nemo_automodel.components.loss.masked_ce.MaskedCrossEntropy

dataset:
  _target_: nemo_automodel.components.datasets.llm.squad.make_squad_dataset
  dataset_name: rajpurkar/squad
  split: train
  limit_dataset_samples: 512

packed_sequence:
  packed_sequence_size: 0

dataloader:
  _target_: torchdata.stateful_dataloader.StatefulDataLoader
  collate_fn: nemo_automodel.components.datasets.utils.default_collater
  shuffle: false

validation_dataset:
  _target_: nemo_automodel.components.datasets.llm.squad.make_squad_dataset
  dataset_name: rajpurkar/squad
  split: validation
  limit_dataset_samples: 64

validation_dataloader:
  _target_: torchdata.stateful_dataloader.StatefulDataLoader
  collate_fn: nemo_automodel.components.datasets.utils.default_collater

optimizer:
  _target_: torch.optim.Adam
  lr: 1.0e-5
  betas: [0.9, 0.999]
  eps: 1.0e-8
  weight_decay: 0

lr_scheduler:
  lr_decay_style: cosine
  min_lr: 1.0e-6

peft:
  _target_: nemo_automodel.components._peft.lora.PeftConfig
  match_all_linear: true
  dim: 8
  alpha: 32
  use_triton: true
EOF

Upload the config to S3.

In [ ]:
if default_prefix:
    input_path = f"{default_prefix}/nemo-automodel"
else:
    input_path = "nemo-automodel"

train_config_s3_path = f"s3://{bucket_name}/{input_path}/config/args.yaml"

s3_client.upload_file("args.yaml", bucket_name, f"{input_path}/config/args.yaml")

print(f"Training config uploaded to:")
print(train_config_s3_path)

---
## Fine-tune Model

We use the SageMaker PyTorch DLC as the base container and install `nemo-automodel` via `requirements.txt`.

This test uses `rajpurkar/squad` from HuggingFace Hub directly, so no S3 data channels are needed — only the config YAML is passed as an input channel.

#### Get PyTorch image_uri

In [ ]:
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session

In [ ]:
sagemaker_session = Session()

bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix

In [ ]:
instance_type = "ml.p5.48xlarge"
instance_count = 1

instance_type

In [ ]:
image_uri = image_uris.retrieve(
    framework="pytorch",
    region=sagemaker_session.boto_session.region_name,
    version="2.8.0",
    instance_type=instance_type,
    image_scope="training"
)

image_uri

In [ ]:
from sagemaker.train.configs import (
    CheckpointConfig,
    Compute,
    OutputDataConfig,
    SourceCode,
    StoppingCondition,
)
from sagemaker.train.distributed import Torchrun
from sagemaker.train.model_trainer import ModelTrainer

# Define the script to be run
source_code = SourceCode(
    source_dir="./scripts",
    requirements="requirements.txt",
    entry_script="train.py",
)

# Define the compute
compute_configs = Compute(
    instance_type=instance_type,
    instance_count=instance_count,
    keep_alive_period_in_seconds=0,
)

# Define Training Job Name
job_name = f"nemo-automodel-{model_id.split('/')[-1].replace('.', '-')}-sft"

# Define OutputDataConfig path
if default_prefix:
    output_path = f"s3://{bucket_name}/{default_prefix}/{job_name}"
else:
    output_path = f"s3://{bucket_name}/{job_name}"

# Define the ModelTrainer
model_trainer = ModelTrainer(
    training_image=image_uri,
    source_code=source_code,
    base_job_name=job_name,
    compute=compute_configs,
    distributed=Torchrun(),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=7200),
    hyperparameters={
        "config": "/opt/ml/input/data/config/args.yaml",
    },
    output_data_config=OutputDataConfig(
        s3_output_path=output_path, compression_type="NONE"
    ),
    checkpoint_config=CheckpointConfig(
        s3_uri=output_path + "/checkpoint", local_path="/opt/ml/checkpoints"
    ),
)

In [ ]:
from sagemaker.train.configs import InputData

# Only the config channel — Squad dataset is downloaded from HuggingFace Hub at runtime
config_input = InputData(
    channel_name="config",
    data_source=train_config_s3_path,
)

data = [config_input]
data

In [ ]:
model_trainer.train(input_data_config=data, wait=False)